## Plot Selected Experiments

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
from pathlib import Path

CSV      = Path("results/xgb_results.csv")   # path to results CSV
OUT      = Path("restuls/images/xgb_selected.png")                       # output path, or None → <csv>_selected.png
MODEL    = "XGBoost"                  # model name shown in figure title
SCENARIO = "drop"                     # "auto", "drop", or "merge"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

SCENARIOS = {
    "drop": {
        "stages": ["Normal cognition", "Subjective Cognitive Decline", "Early MCI", "Late MCI", "Dementia"],
        "stages_short": ["NC", "SCD", "EMCI", "LMCI", "Dem"],
        "title_suffix": "[NC, SCD, EMCI, LMCI, DEM]",
    },
    "merge": {
        "stages": ["Normal cognition", "SCD", "Early Impaired", "Late MCI", "Dementia"],
        "stages_short": ["NC", "SCD", "EImp", "LMCI", "Dem"],
        "title_suffix": "[NC, SCD, EImp, LMCI, DEM]",
    },
}

TEXT_COLOR = "black"

In [ ]:
# ── Load & filter ─────────────────────────────────────────────────────────────
df = pd.read_csv(CSV).sort_values("f1_macro_mean", ascending=False).reset_index(drop=True)

print("Available experiments:")
for name in df["experiment"]:
    print(" ", name)

In [ ]:
EXPERIMENTS = [
    "E35_CDRSUM_HVLTDR_PLASMA",
    "E1_CDRSUM",
    "E9_CDRSUM_HVLTDR",
    "E77_CDRSUM_HVLTDR_PLASMA_APOE"
]

In [ ]:
mask = pd.Series(False, index=df.index)
for ident in EXPERIMENTS:
    mask |= df["experiment"].str.contains(ident, case=False, regex=False)
df_sel = df[mask].reset_index(drop=True)

print(f"Matched {len(df_sel)} experiment(s):")
for name in df_sel["experiment"]:
    print(" ", name)

In [ ]:
# ── Detect scenario ───────────────────────────────────────────────────────────
if SCENARIO == "auto":
    cols = set(df_sel.columns)
    scenario_name = next(
        name for name, cfg in SCENARIOS.items()
        if {f"precision_{s}_mean" for s in cfg["stages"]}.issubset(cols)
    )
else:
    scenario_name = SCENARIO

cfg          = SCENARIOS[scenario_name]
stages       = cfg["stages"]
stages_short = cfg["stages_short"]
print(f"Scenario: {scenario_name}  →  {stages_short}")

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
n = len(df_sel)
fig = plt.figure(figsize=(16, 8))
gs  = GridSpec(1, 3, figure=fig, width_ratios=[2.2, 1, 1], wspace=0.45)

ax_overall = fig.add_subplot(gs[0, 0])
ax_recall  = fig.add_subplot(gs[0, 1])
ax_prec    = fig.add_subplot(gs[0, 2])

# ── Panel A: overall metrics ──────────────────────────────────────────────────
y = np.arange(n)
h = 0.4

ax_overall.barh(y - h/2, df_sel["f1_macro_mean"], height=h,
                xerr=df_sel["f1_macro_std"], label="Macro-F1",
                color="#2E86AB", ecolor="#1a4d63", capsize=2, alpha=0.9)
ax_overall.barh(y + h/2, df_sel["balanced_accuracy_mean"], height=h,
                xerr=df_sel["balanced_accuracy_std"], label="Balanced accuracy",
                color="#E07A5F", ecolor="#8a3f2a", capsize=2, alpha=0.9)

ax_overall.set_yticks(y)
ax_overall.set_yticklabels(
    [f"{e}  (k={k})" for e, k in zip(df_sel["experiment"], df_sel["n_features"])],
    fontsize=8, color=TEXT_COLOR,
)
ax_overall.invert_yaxis()
ax_overall.set_xlim(0, 1.2)
ax_overall.set_xlabel("Score (mean ± std over 4 CV folds)", color=TEXT_COLOR)
ax_overall.set_title("A. Overall performance per experiment", loc="left",
                     fontweight="bold", color=TEXT_COLOR)
chance = 1.0 / len(stages)
ax_overall.axvline(chance, color="grey", lw=0.5, ls=":", alpha=0.6)
ax_overall.text(chance, -0.45, "chance", fontsize=7, color=TEXT_COLOR, ha="center")
legend = ax_overall.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.08),
    framealpha=0.9, fontsize=9, ncol=2,
)
for txt in legend.get_texts():
    txt.set_color(TEXT_COLOR)
for i, row in df_sel.iterrows():
    f1_whisker  = row["f1_macro_mean"] + row["f1_macro_std"]
    ba_whisker  = row["balanced_accuracy_mean"] + row["balanced_accuracy_std"]
    ax_overall.text(f1_whisker + 0.01, i - h/2,
                    f"{row['f1_macro_mean']:.3f}±{row['f1_macro_std']:.3f}",
                    va="center", ha="left", fontsize=7, color="#2E86AB")
    ax_overall.text(ba_whisker + 0.01, i + h/2,
                    f"{row['balanced_accuracy_mean']:.3f}±{row['balanced_accuracy_std']:.3f}",
                    va="center", ha="left", fontsize=7, color="#E07A5F")
ax_overall.tick_params(axis="x", colors=TEXT_COLOR)
ax_overall.tick_params(axis="y", colors=TEXT_COLOR)
ax_overall.grid(axis="x", alpha=0.3)

# ── Panels B & C: heatmaps ────────────────────────────────────────────────────
def short_id(name):
    return name.split("_", 1)[0]

def plot_heatmap(ax, metric, title, cmap):
    cols   = [f"{metric}_{s}_mean" for s in stages]
    matrix = df_sel[cols].to_numpy()
    im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=1.15)
    ax.set_xticks(np.arange(len(stages_short)))
    ax.set_xticklabels(stages_short, fontsize=9, color=TEXT_COLOR)
    ax.set_yticks(np.arange(n))
    ax.set_yticklabels([short_id(e) for e in df_sel["experiment"]],
                       fontsize=8, color=TEXT_COLOR)
    ax.set_title(title, loc="left", fontweight="bold", color=TEXT_COLOR)
    ax.tick_params(axis="x", colors=TEXT_COLOR)
    ax.tick_params(axis="y", colors=TEXT_COLOR)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center",
                    color=TEXT_COLOR, fontsize=7)
    # cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    # cbar.ax.tick_params(labelsize=8, colors=TEXT_COLOR)
    # cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])

plot_heatmap(ax_recall, "recall",    "B. Recall per class",    cmap="Oranges")
plot_heatmap(ax_prec,   "precision", "C. Precision per class", cmap="Blues")

title = f"{MODEL} — feature-set comparison — {cfg['title_suffix']}"
fig.suptitle(title, fontsize=13, fontweight="bold", y=0.995, color=TEXT_COLOR)
fig.tight_layout(rect=[0, 0, 1, 1])

out_path = Path(OUT) if OUT else CSV.with_name(f"{CSV.stem}_selected.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")